# 04 · Final model per target and the external test (once)

1. For each target, the model chosen in notebook 03 is tuned on **all** development boreholes (inner groups = the 5 outer folds of the 106 m block) and trained on all of them, then saved (frozen).
2. It is scored **once** on V3's 11 external-holdout boreholes, beside the depth reference. A `_soil` model gets the soil type from the thesis's **final classifier** (V3-clean notebook 05, frozen; folder in `settings.toml → [soil_type] v3clean_folder`), and is also scored with the logged type for comparison.

A lock file (`outputs/04_final_external/private/external_test_done.json`) stops a second external test: if it exists, the notebook only shows the saved scores. Delete it only for a documented reason, and report that in the thesis.

**Run time:** about 5-15 minutes per target.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
import joblib

PRIV, O = H.out_dir(S, "04_final_external", private=True), H.out_dir(S, "04_final_external")
LOCK = PRIV / "external_test_done.json"
main_block = H.block_sizes(S)[0]
choice = pd.read_csv(H.out_dir(S, "03_results") / "model_choice.csv").set_index("target").selected
display(choice)

## Final models (frozen)

In [ ]:
for t, name in choice.items():
    f = PRIV / f"final_{t}_{name}.joblib"
    if f.exists():
        print(t, name, "already trained (frozen)"); continue
    d = pd.read_csv(H.processed_dir(S) / f"samples_{t}.csv")
    res = H.fit_final(name, t, d[d.set == "development"], main_block, S)
    joblib.dump({"model": res["model"], "params": res["params"], "inner_rmse": res["inner_rmse"]}, f, compress=3)
    print(t, name, "trained; inner RMSE", None if np.isnan(res["inner_rmse"]) else round(res["inner_rmse"], 3))
    if name != "Depth":     # the depth reference is always scored beside the final model
        d = d[d.set == "development"]
        joblib.dump({"model": H.make_model("Depth", {}, 0, t, S).fit(d[H.FEATURES].values, d.y.values, d.depth_m.values)},
                    PRIV / f"final_{t}_Depth.joblib", compress=3)

## External test (runs once)

In [ ]:
if LOCK.exists():
    print("The external test was already run on", json.loads(LOCK.read_text())["date"], "- showing the saved scores.")
else:
    rows, preds = [], []
    for t, name in choice.items():
        d = pd.read_csv(H.processed_dir(S) / f"samples_{t}.csv")
        ext = d[d.set == "external"]
        for mod in dict.fromkeys([name, "Depth"]):
            model = joblib.load(PRIV / f"final_{t}_{mod}.joblib")["model"]
            variants = [("none", None)]
            if H.uses_soil(mod):
                if "CLF" not in globals():
                    CLF = H.load_classifier(S)
                    print("Classifier for the soil type:", CLF[2], "| threshold", CLF[1])
                variants = [("predicted", H.classify(ext, CLF[0], CLF[1], S)), ("logged", None)]
            for st, soil in variants:
                p = model.predict(H.design(ext, mod, soil), ext.depth_m.values) if len(ext) else np.array([])
                preds.append(ext[["BH_Code", "depth_m", "y"]].assign(y_pred=p, target=t, model=mod, soil_type=st, final=mod == name))
                rows.append({"target": t, "model": mod, "soil_type": st, "final_model": mod == name,
                             "boreholes": ext.BH_Code.nunique(), **H.scores(ext.y, p)})
    pd.concat(preds).to_csv(PRIV / "external_predictions.csv", index=False)
    pd.DataFrame(rows).to_csv(O / "external_scores.csv", index=False)
    LOCK.write_text(json.dumps({"date": time.strftime("%Y-%m-%d %H:%M"), "models": choice.to_dict()}))
display(pd.read_csv(O / "external_scores.csv").round(3))
print(f"Finished in {time.time() - T0:.0f} s.")